# Commodity Delivery Gap Analysis
## Oil Tanker Imports by US PADD + Derivative Commodities (Helium, NGL)

**Data Sources:**
- **EIA API v2** — Crude oil imports by PADD/port, petroleum movements by transport mode
- **EIA STEO** — Short Term Energy Outlook (forward projections)
- **USGS MCS** — Helium supply/demand (annual, synthetic here)
- **AISstream.io** — Real-time vessel tracking (tanker positions + ETAs)
- **Yahoo Finance** — Commodity futures (WTI, NatGas, RBOB, Heating Oil)

**Key Thesis:** Helium has no substitute and is extracted as a byproduct of natural gas processing. NatGas import/production disruptions are a leading indicator for helium supply gaps.

In [ ]:
# --- Colab Setup (skip if running locally) ---
import sys
if 'google.colab' in sys.modules:
    !pip install -q git+https://github.com/hb-cam/commodity-flow-intelligence.git
    import os
    from getpass import getpass
    # Optional: enter your EIA API key for live data (press Enter to skip)
    _eia_key = getpass('EIA API Key (Enter to skip): ')
    if _eia_key:
        os.environ['EIA_API_KEY'] = _eia_key
        os.environ['USE_LIVE_API'] = 'true'
    _ais_key = getpass('AISstream API Key (Enter to skip): ')
    if _ais_key:
        os.environ['AISSTREAM_API_KEY'] = _ais_key
    print('Setup complete.' + (' Live mode enabled.' if _eia_key else ' Using synthetic data.'))

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import numpy as np
import pandas as pd
from datetime import timedelta
import warnings
warnings.filterwarnings('ignore')

from commodity_flow import config, eia, synthetic, analysis, futures
from commodity_flow.provenance import ProvenanceTracker, DataSource

plt.style.use(config.PLOT_STYLE)
plt.rcParams['figure.figsize'] = config.PLOT_FIGSIZE
plt.rcParams['font.size'] = config.PLOT_FONTSIZE

prov = ProvenanceTracker()

print(f"Live API mode: {config.USE_LIVE_API}")
print(f"EIA API key: {'configured' if config.EIA_API_KEY else 'not set (using synthetic data)'}")

---
## 1. Load Data

Toggle `USE_LIVE_API` in `.env` to switch between synthetic and live EIA data.

In [ ]:
if config.USE_LIVE_API and config.EIA_API_KEY:
    print("Pulling from EIA API v2...")
    df_imports = eia.fetch_crude_imports_by_padd(config.EIA_API_KEY)
    prov.record(DataSource("Crude imports", "EIA API v2", "petroleum/move/imp", True,
        rows=len(df_imports), date_range=f"{df_imports['date'].min().date()} to {df_imports['date'].max().date()}"))

    df_stocks = eia.fetch_weekly_stocks(config.EIA_API_KEY)
    prov.record(DataSource("Weekly stocks", "EIA API v2", "petroleum/stoc/wstk", True,
        rows=len(df_stocks), date_range=f"{df_stocks['date'].min().date()} to {df_stocks['date'].max().date()}",
        notes="Total petroleum by PADD (crude-only not available at PADD level)"))

    df_steo = eia.fetch_steo_projections(config.EIA_API_KEY)
    prov.record(DataSource("STEO projections", "EIA API v2", "steo", True,
        rows=len(df_steo), notes="CONIPUS (net imports), COPRPUS (production), PAPR_WORLD"))

    df_natgas = eia.fetch_natgas_imports(config.EIA_API_KEY)
    prov.record(DataSource("NatGas imports", "EIA API v2", "natural-gas/move/poe1", True,
        rows=len(df_natgas), date_range=f"{df_natgas['date'].min().date()} to {df_natgas['date'].max().date()}",
        notes="Pipeline + LNG split at national level (Bcf)"))

    print(f"Imports: {len(df_imports)} | Stocks: {len(df_stocks)} | STEO: {len(df_steo)} | NatGas: {len(df_natgas)}")
else:
    print("Using synthetic data (set USE_LIVE_API=true in .env for real data)")
    df_imports = synthetic.generate_synthetic_imports()
    prov.record(DataSource("Crude imports", "Synthetic", "synthetic generator", False, rows=len(df_imports)))

    df_stocks = synthetic.generate_synthetic_stocks()
    prov.record(DataSource("Weekly stocks", "Synthetic", "synthetic generator", False, rows=len(df_stocks)))

    df_steo = synthetic.generate_synthetic_steo()
    prov.record(DataSource("STEO projections", "Synthetic", "synthetic generator", False, rows=len(df_steo)))

    df_natgas = synthetic.generate_synthetic_natgas_imports()
    prov.record(DataSource("NatGas imports", "Synthetic", "synthetic generator", False, rows=len(df_natgas),
        notes="US is now net LNG exporter; synthetic LNG imports are near-zero"))

# Always synthetic (annual PDF sources, no API)
df_helium = synthetic.generate_synthetic_helium()
prov.record(DataSource("Helium supply/demand", "Synthetic (USGS MCS)", "synthetic generator", False,
    rows=len(df_helium), notes="USGS publishes annual PDFs, no API available"))

print(f"\nImports: {df_imports.shape} | Stocks: {df_stocks.shape}")
print(f"NatGas: {df_natgas.shape} | Helium: {df_helium.shape} | STEO: {df_steo.shape}")
df_imports.head(3)

---
### Data Provenance

Run the cell below to see which datasets are live vs synthetic, row counts, and date ranges.

In [ ]:
from IPython.display import Markdown
display(Markdown(prov.summary()))

---
## 2. Crude Oil Import Gap Analysis by PADD

**Method:** Compare trailing 12-month average to recent months. A "gap" = actual imports below the trailing norm by > 1 std dev.

In [ ]:
fig, axes = plt.subplots(3, 2, figsize=(16, 14), sharex=True)
axes = axes.flatten()

gap_summary = []

for i, (padd, name) in enumerate(config.PADDS.items()):
    ax = axes[i]
    sub = df_imports[df_imports["duoarea"] == padd].sort_values("date").copy()
    sub = analysis.detect_gaps(sub, column="value", window=12, threshold=-1.0)

    # Trailing stats for plotting
    sub["ma12"] = sub["value"].rolling(12, min_periods=6).mean()
    sub["std12"] = sub["value"].rolling(12, min_periods=6).std()
    sub["lower_band"] = sub["ma12"] - sub["std12"]

    ax.plot(sub["date"], sub["value"], 'o-', ms=3, label="Actual imports", color="#2563eb")
    ax.plot(sub["date"], sub["ma12"], '--', label="12-mo avg", color="#64748b", lw=2)
    ax.fill_between(sub["date"], sub["lower_band"], sub["ma12"],
                     alpha=0.15, color="#64748b", label="Normal band")

    # Highlight gap periods
    for gd in sub[sub["in_gap"]]["date"]:
        ax.axvspan(gd - timedelta(days=15), gd + timedelta(days=15),
                   alpha=0.25, color="#ef4444")

    ax.set_title(f"{padd}: {name}", fontweight="bold")
    ax.set_ylabel("Thousand Barrels")
    ax.legend(fontsize=8, loc="lower left")

    # Summary stats
    recent = sub[sub["date"] >= "2025-09-01"]
    if recent["in_gap"].any():
        gap_months = recent[recent["in_gap"]]["date"].dt.strftime("%Y-%m").tolist()
        avg_deficit = recent[recent["in_gap"]]["z_score"].mean()
        gap_summary.append({
            "padd": padd, "region": name,
            "gap_months": ", ".join(gap_months),
            "avg_z_score": round(avg_deficit, 2),
        })

axes[5].set_visible(False)
fig.suptitle("Crude Oil Imports by PADD \u2014 Delivery Gap Detection",
             fontsize=16, fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()

if gap_summary:
    print("\n=== DETECTED DELIVERY GAPS (recent) ===")
    display(pd.DataFrame(gap_summary))
else:
    print("No significant delivery gaps detected in recent window.")

---
## 3. Weekly Stocks Drawdown — Days-of-Supply Risk

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 9), sharey=False)
axes = axes.flatten()

for i, (padd, name) in enumerate(config.PADDS.items()):
    ax = axes[i]
    sub = df_stocks[df_stocks["duoarea"] == padd].sort_values("date").copy()
    sub["delta_4w"] = sub["value"].diff(4)

    color = sub["delta_4w"].apply(lambda x: "#ef4444" if x < 0 else "#22c55e")
    ax.bar(sub["date"], sub["delta_4w"], width=5, color=color, alpha=0.7)
    ax.axhline(0, color="black", lw=0.5)
    ax.set_title(f"{padd}: {name}\n4-Week Stock Change", fontweight="bold")
    ax.set_ylabel("MBBL change")
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))
    ax.tick_params(axis='x', rotation=45)

axes[5].set_visible(False)
fig.suptitle("Weekly Crude Stock Changes by PADD \u2014 Drawdown Alert",
             fontsize=16, fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()

---
## 4. Natural Gas Imports — Helium Supply Leading Indicator

Helium is extracted during natural gas processing (cryogenic separation).  
**No natural gas throughput → no helium production.** Pipeline + LNG disruptions cascade to helium supply.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

for mode, color, ax in [("Pipeline", "#2563eb", ax1), ("LNG", "#7c3aed", ax2)]:
    sub = df_natgas[df_natgas["mode"] == mode].sort_values("date")
    sub["ma6"] = sub["value_bcf"].rolling(6, min_periods=3).mean()
    sub["std6"] = sub["value_bcf"].rolling(6, min_periods=3).std()

    ax.plot(sub["date"], sub["value_bcf"], 'o-', ms=3, color=color, label=f"{mode} imports")
    ax.plot(sub["date"], sub["ma6"], '--', color="#64748b", label="6-mo avg")
    ax.fill_between(sub["date"], sub["ma6"] - sub["std6"], sub["ma6"] + sub["std6"],
                     alpha=0.1, color="#64748b")

    gap_mask = sub["value_bcf"] < (sub["ma6"] - 1.5 * sub["std6"])
    ax.scatter(sub[gap_mask]["date"], sub[gap_mask]["value_bcf"],
               color="#ef4444", s=80, zorder=5, label="Disruption", marker="v")

    ax.set_title(f"Natural Gas Imports \u2014 {mode}", fontweight="bold")
    ax.set_ylabel("Billion Cubic Feet (Bcf)")
    ax.legend(fontsize=9)

fig.suptitle("Natural Gas Import Disruptions \u2192 Helium Supply Chain Risk",
             fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

---
## 5. Helium Supply-Demand Gap + Price Signal

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

x = df_helium["year"]
ax1.bar(x - 0.2, df_helium["world_production_Mcm"], 0.35, label="World Production", color="#2563eb")
ax1.bar(x + 0.2, df_helium["world_demand_Mcm"], 0.35, label="World Demand", color="#f97316")
ax1.axhline(0, color="black", lw=0.5)

for _, row in df_helium[df_helium["supply_gap_Mcm"] < 0].iterrows():
    ax1.axvspan(row["year"] - 0.45, row["year"] + 0.45, alpha=0.15, color="#ef4444")

ax1.set_title("Global Helium Supply vs. Demand", fontweight="bold")
ax1.set_ylabel("Million Cubic Meters")
ax1.legend()
ax1.set_xticks(x)

ax2.plot(x, df_helium["blm_price_usd_per_mcf"], 'o-', color="#dc2626", lw=2, ms=7)
ax2.fill_between(x, df_helium["blm_price_usd_per_mcf"], alpha=0.1, color="#dc2626")
ax2.set_title("Helium Price (BLM Grade-A)", fontweight="bold")
ax2.set_ylabel("USD per Mcf")
ax2.set_xticks(x)

ax2.annotate("Shortage pricing\nkicks in", xy=(2023, 35),
             xytext=(2021, 38), fontsize=10,
             arrowprops=dict(arrowstyle="->", color="#dc2626"),
             color="#dc2626", fontweight="bold")

fig.suptitle("Helium: No Substitutes, Constrained Supply, Surging Demand (MRI, Semiconductors, Quantum)",
             fontsize=13, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

print("\n=== Helium Supply Gap Summary ===")
display(df_helium[["year", "supply_gap_Mcm", "blm_price_usd_per_mcf"]].tail(5))

---
## 6. Composite Gap Scorecard + STEO Forward Overlay

Combines oil + natural gas z-scores into a single delivery-risk metric.  
STEO forward projections extend the scorecard into forecast territory (dashed).

In [ ]:
scorecard = analysis.build_scorecard(df_imports, df_natgas, df_steo)

fig, ax = plt.subplots(figsize=(16, 6))

# Split actual vs forecast
actual = scorecard[~scorecard["is_forecast"]]
forecast = scorecard[scorecard["is_forecast"]]

ax.plot(actual.index, actual["oil_import_z"], label="Oil Import Z", color="#2563eb", lw=1.5)
ax.plot(actual.index, actual["natgas_import_z"], label="NatGas Import Z", color="#7c3aed", lw=1.5)
ax.plot(actual.index, actual["composite_gap_score"], label="Composite Gap Score",
        color="#dc2626", lw=2.5)

# STEO forecast extension
if not forecast.empty:
    # Connect actual to forecast
    bridge = pd.concat([actual.tail(1), forecast])
    ax.plot(bridge.index, bridge["composite_gap_score"], '--',
            color="#dc2626", lw=1.5, alpha=0.6, label="STEO Forecast")
    ax.axvspan(forecast.index.min(), forecast.index.max(),
               alpha=0.05, color="#fbbf24", label="Forecast zone")

ax.axhline(-1, color="#f97316", ls="--", label="Warning (-1\u03c3)")
ax.axhline(-2, color="#ef4444", ls="--", label="Critical (-2\u03c3)")
ax.axhline(0, color="black", lw=0.5)

ax.fill_between(actual.index, -2, actual["composite_gap_score"],
                where=actual["composite_gap_score"] < -2,
                alpha=0.3, color="#ef4444", interpolate=True)

ax.set_title("Composite Delivery Gap Scorecard \u2014 Oil + Natural Gas (Helium Proxy) + STEO Forward",
             fontsize=14, fontweight="bold")
ax.set_ylabel("Z-Score (negative = below normal)")
ax.legend(loc="lower left", fontsize=9)
plt.tight_layout()
plt.show()

# Alert table
alerts = actual[actual["composite_gap_score"] < -1].tail(10)
if not alerts.empty:
    print("\n=== MONTHS WITH GAP ALERTS (composite < -1\u03c3) ===")
    display(alerts.round(2))

---
## 7. Futures Price Overlay — Physical vs. Market Divergence

Compare physical delivery gap z-scores with commodity futures price z-scores.  
**Divergence** = physical supply is tighter/looser than the market is pricing.

In [ ]:
try:
    df_fut = futures.fetch_futures_curves()
    df_fut = futures.compute_futures_z_scores(df_fut)
    has_futures = not df_fut.empty
except Exception as e:
    print(f"Futures data unavailable: {e}")
    has_futures = False

if has_futures:
    symbols = df_fut["symbol"].unique()
    fig, axes = plt.subplots(1, len(symbols), figsize=(6 * len(symbols), 5), sharey=True)
    if len(symbols) == 1:
        axes = [axes]

    colors = {"CL=F": "#2563eb", "NG=F": "#7c3aed", "RB=F": "#16a34a", "HO=F": "#ea580c"}

    for ax, sym in zip(axes, symbols):
        sub = df_fut[df_fut["symbol"] == sym].sort_values("date")
        name = sub["name"].iloc[0]
        c = colors.get(sym, "#64748b")

        ax.plot(sub["date"], sub["futures_z"], color=c, lw=1.5, label=f"{name} Z")
        ax.axhline(-1, color="#f97316", ls="--", lw=0.8)
        ax.axhline(1, color="#22c55e", ls="--", lw=0.8)
        ax.axhline(0, color="black", lw=0.5)
        ax.fill_between(sub["date"], -1, sub["futures_z"],
                        where=sub["futures_z"] < -1, alpha=0.2, color="#ef4444", interpolate=True)
        ax.fill_between(sub["date"], 1, sub["futures_z"],
                        where=sub["futures_z"] > 1, alpha=0.2, color="#22c55e", interpolate=True)
        ax.set_title(f"{name} ({sym})", fontweight="bold")
        ax.set_ylabel("Price Z-Score" if ax == axes[0] else "")
        ax.tick_params(axis='x', rotation=30)

    fig.suptitle("Commodity Futures Z-Scores \u2014 Compare with Physical Gap Scorecard",
                 fontsize=14, fontweight="bold", y=1.02)
    plt.tight_layout()
    plt.show()
else:
    print("Skipping futures overlay (no data available).")

---
## 8. AIS Tanker Tracker

Connects to AISstream.io websocket to collect real-time tanker positions near US ports.  
**Requires:** AISstream.io API key in `.env`.

In [ ]:
from commodity_flow import ais
import asyncio

if config.AISSTREAM_API_KEY:
    print("Connecting to AISstream.io (60s collection window)...")
    df_tankers = await ais.track_tankers(
        config.AISSTREAM_API_KEY,
        duration_seconds=60,
        max_positions=200,
    )
    print(f"Collected {len(df_tankers)} tanker positions.")

    if not df_tankers.empty:
        fig, ax = plt.subplots(figsize=(14, 8))
        scatter = ax.scatter(
            df_tankers["lon"], df_tankers["lat"],
            c=df_tankers["speed_kn"], cmap="RdYlGn_r",
            s=20, alpha=0.7, edgecolors="none"
        )
        plt.colorbar(scatter, label="Speed (knots)")

        # Draw bounding boxes
        for box in ais.US_PORT_BOXES:
            sw, ne = box
            ax.plot([sw[1], ne[1], ne[1], sw[1], sw[1]],
                    [sw[0], sw[0], ne[0], ne[0], sw[0]],
                    'b--', alpha=0.3, lw=1)

        ax.set_xlabel("Longitude")
        ax.set_ylabel("Latitude")
        ax.set_title(f"Tanker Positions ({len(df_tankers)} reports)", fontweight="bold")
        plt.tight_layout()
        plt.show()

        display(df_tankers.head(10))
else:
    print("AISstream.io API key not set. Set AISSTREAM_API_KEY in .env to enable.")
    print("Register free at: https://aisstream.io")

---
## 9. Derivative Commodity Map

Commodities whose supply chains are measurable via shipping/import data and have tradeable derivatives.

In [ ]:
commodity_map = pd.DataFrame([
    {"commodity": "Crude Oil",
     "instruments": "WTI/Brent futures, crack spreads, PADD basis",
     "data_source": "EIA API v2 (petroleum/move/imp, crude-oil-imports)",
     "shipping_trackable": True,
     "lead_time": "2-6 weeks (AIS tanker ETA)",
     "gap_signal": "PADD import shortfall vs 12-mo avg"},
    {"commodity": "Helium",
     "instruments": "No direct futures; equities (DME, RLPI, PHX)",
     "data_source": "USGS MCS (annual); BLM auction prices",
     "shipping_trackable": False,
     "lead_time": "Lagging (annual); proxy via NatGas throughput",
     "gap_signal": "NatGas disruptions \u2192 helium co-production collapse"},
    {"commodity": "Natural Gas (LNG)",
     "instruments": "Henry Hub, JKM (Asia LNG), TTF (Europe)",
     "data_source": "EIA API v2 (natural-gas/move), FERC LNG reports",
     "shipping_trackable": True,
     "lead_time": "2-4 weeks (LNG carrier AIS)",
     "gap_signal": "LNG import volume vs seasonal norm"},
    {"commodity": "Gasoline / Distillates",
     "instruments": "RBOB futures, heating oil futures, crack spreads",
     "data_source": "EIA API v2 (petroleum/move/wkly, petroleum/stoc/wstk)",
     "shipping_trackable": True,
     "lead_time": "1-3 weeks (product tanker AIS)",
     "gap_signal": "Weekly stocks drawdown rate by PADD"},
    {"commodity": "Propane / NGLs",
     "instruments": "Mt. Belvieu propane, ethane/butane swaps",
     "data_source": "EIA API v2 (petroleum/stoc, natural-gas/move)",
     "shipping_trackable": True,
     "lead_time": "1-2 weeks",
     "gap_signal": "NGL stock levels + fractionation utilization"},
    {"commodity": "Lithium",
     "instruments": "CME lithium hydroxide futures, equity (ALB, SQM)",
     "data_source": "USGS MCS, Fastmarkets, Benchmark Minerals",
     "shipping_trackable": True,
     "lead_time": "4-8 weeks (bulk carrier AIS from Chile/Australia)",
     "gap_signal": "Port departures vs contract volumes"},
])

display(commodity_map.style.set_properties(**{
    'text-align': 'left',
    'white-space': 'pre-wrap',
}).set_table_styles([
    {'selector': 'th', 'props': [('text-align', 'left'), ('font-weight', 'bold')]}
]))

In [ ]:
# Data provenance footnotes for this session
print("\n" + "="*70)
print("DATA PROVENANCE")
print("="*70)
for fn in prov.footnotes():
    print(f"  {fn}")
print()